In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import operator
from typing import Annotated,TypedDict,List
from langgraph.graph import StateGraph, END
from IPython.display import Image, display
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

class State(TypedDict):
    messages: Annotated[List[str], operator.add]

builder=StateGraph(State)

def chat_with_model(state):
    print(state)
    print("--------------------")
    messages=state['messages']
    response=llm.invoke(messages)
    return {"messages":[response]}

def convert_messages(state):
    EXTRACTION_PROMPT= """
    你是一位数据提取专家，负责从文本中检索关键信息，请为所提供的文本提取相关信息，并以JSON格式输出，概述所提取的关键数据点。
    """
    print(state)
    print("---------------------------")
    messages=state['messages']
    messages=messages[-1]

    messages=[
        SystemMessage(content=EXTRACTION_PROMPT),
        HumanMessage(content=state['messages'][-1].content)
    ]
    response=llm.invoke(messages)
    return {"messages":[response]}

builder.add_node("chat_with_model",chat_with_model)
builder.add_node("convert_messages",convert_messages)

builder.set_entry_point("chat_with_model")

builder.add_edge("chat_with_model","convert_messages")
builder.add_edge("convert_messages",END)

graph=builder.compile()

In [ ]:
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
query="你好，请你介绍一下你自己"
input_message= {"messages":[HumanMessage(content=query)]}
result=graph.invoke(input_message)
print(result)

{'messages': [HumanMessage(content='你好，请你介绍一下你自己', additional_kwargs={}, response_metadata={})]}
--------------------
{'messages': [HumanMessage(content='你好，请你介绍一下你自己', additional_kwargs={}, response_metadata={}), AIMessage(content='你好呀！很高兴认识你！😊\n\n我是**DeepSeek**，由深度求索公司创造的AI助手。让我给你介绍一下我的“特长”：\n\n**我能做什么？**\n- 📝 **文字处理**：写作、翻译、总结、改写等各种文本任务\n- 💡 **知识问答**：回答各类问题，从科学到历史，从技术到生活\n- 📄 **文件阅读**：支持上传图片、PDF、Word、Excel、PPT等文件，帮你提取和分析信息\n- 🔗 **链接阅读**：可以读取网页链接内容（需要你手动开启联网搜索功能）\n- 🧠 **深度思考**：擅长逻辑推理、代码编写、复杂问题分析\n\n**我的特点：**\n- ✅ **完全免费**：没有收费计划，放心使用\n- 📚 **超大上下文**：1M token容量，可以一次性处理像《三体》三部曲那么长的内容\n- 🎤 **语音输入**：App端支持语音交互\n- 🌐 **联网搜索**：需要你在Web/App手动开启\n\n**我的小限制：**\n- 纯文本模型，不能直接“看”图片，但可以读取图片中的文字\n- 知识截止于2025年5月\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！🚀', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 282, 'prompt_tokens': 9, 'total_tokens': 291, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '26bb0e1f-946c-4965-abec-bbd8767452da', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eaf84-7200-74e1-8b3c-ba2495880ba9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 282, 'total_tokens': 291, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}
---------------------------
{'messages': [HumanMessage(content='你好，请你介绍一下你自己', additional_kwargs={}, response_metadata={}), AIMessage(content='你好呀！很高兴认识你！😊\n\n我是**DeepSeek**，由深度求索公司创造的AI助手。让我给你介绍一下我的“特长”：\n\n**我能做什么？**\n- 📝 **文字处理**：写作、翻译、总结、改写等各种文本任务\n- 💡 **知识问答**：回答各类问题，从科学到历史，从技术到生活\n- 📄 **文件阅读**：支持上传图片、PDF、Word、Excel、PPT等文件，帮你提取和分析信息\n- 🔗 **链接阅读**：可以读取网页链接内容（需要你手动开启联网搜索功能）\n- 🧠 **深度思考**：擅长逻辑推理、代码编写、复杂问题分析\n\n**我的特点：**\n- ✅ **完全免费**：没有收费计划，放心使用\n- 📚 **超大上下文**：1M token容量，可以一次性处理像《三体》三部曲那么长的内容\n- 🎤 **语音输入**：App端支持语音交互\n- 🌐 **联网搜索**：需要你在Web/App手动开启\n\n**我的小限制：**\n- 纯文本模型，不能直接“看”图片，但可以读取图片中的文字\n- 知识截止于2025年5月\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！🚀', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 282, 'prompt_tokens': 9, 'total_tokens': 291, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '26bb0e1f-946c-4965-abec-bbd8767452da', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eaf84-7200-74e1-8b3c-ba2495880ba9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 282, 'total_tokens': 291, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}), AIMessage(content='{\n  "名称": "DeepSeek",\n  "开发者": "深度求索公司",\n  "类型": "AI助手",\n  "功能": [\n    "文字处理：写作、翻译、总结、改写",\n    "知识问答：回答科学、历史、技术、生活等问题",\n    "文件阅读：支持图片、PDF、Word、Excel、PPT等文件提取和分析",\n    "链接阅读：读取网页链接内容（需手动开启联网搜索）",\n    "深度思考：逻辑推理、代码编写、复杂问题分析"\n  ],\n  "特点": [\n    "完全免费，无收费计划",\n    "超大上下文：1M token容量",\n    "语音输入：App端支持",\n    "联网搜索：需在Web/App手动开启"\n  ],\n  "限制": [\n    "纯文本模型，不能直接识别图片，但可读取图片中的文字",\n    "知识截止于2025年5月"\n  ]\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 200, 'prompt_tokens': 322, 'total_tokens': 522, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 322}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '9d83e556-db54-4899-8422-6cf01068f7df', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eaf84-8040-73f1-aec6-d7d16a8dfda3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 322, 'output_tokens': 200, 'total_tokens': 522, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}

In [ ]:
print(result["messages"][0].content)
print(result["messages"][1].content)
print(result["messages"][2].content)